# Darwin-ST 时空预测实验分析 (Spatio-Temporal Experiment Analysis)

分析自主演化循环产出的时空预测实验结果。指标为 **masked MAE / RMSE / MAPE**（越低越好），
对照 `baseline_registry.py` 中目标数据集的 SOTA 基线，追踪"逼近并超越 SOTA"的演化轨迹。

> 数据来源：`results.tsv`（每轮一行）。列见下方。后续 P1 阶段会迁移到结构化 SQLite 试验库
> （见 `docs/BLUEPRINT.md` §3），届时本 notebook 直接读 DB 即可。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取实验日志 (tab-separated)。
# 列约定 (见 BLUEPRINT §3 的 experiments schema 精简版):
#   commit  dataset  val_mae  val_rmse  val_mae_std  status  description
# 注: val_mae 为 masked MAE(已 inverse-transform 回真实交通流尺度), 越低越好。
df = pd.read_csv("results.tsv", sep="\t")

for col in ["val_mae", "val_rmse", "val_mae_std"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

# 当前实验针对的数据集(用于拉取对应 SOTA 基线)
DATASET = df["dataset"].dropna().iloc[0] if "dataset" in df.columns and df["dataset"].notna().any() else "PeMS04"

print(f"目标数据集 (Dataset): {DATASET}")
print(f"总实验轮数 (Total experiments): {len(df)}")
print(f"列 (Columns): {list(df.columns)}")
df.head(10)

In [ ]:
# 实验结局统计 (Outcome counts)
# 状态: KEEP(强化保留) / DISCARD(劣化舍弃) / CRASH(数值崩溃) / PRUNED(多保真早停淘汰)
counts = df["status"].value_counts()
print("实验结局 (Experiment outcomes):")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_pruned = counts.get("PRUNED", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\n保留率 (Keep rate): {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")
if n_crash or n_pruned:
    print(f"崩溃 {n_crash} 次 / 早停淘汰 {n_pruned} 次 (算力节省信号)")

In [ ]:
# 列出所有 KEPT 实验 (留存下来的精度改进)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT 实验 ({len(kept)} 条):\n")
for i, row in kept.iterrows():
    mae = row["val_mae"]
    rmse = row.get("val_rmse", float("nan"))
    std = row.get("val_mae_std", float("nan"))
    desc = row["description"]
    std_str = f"±{std:.3f}" if pd.notna(std) else ""
    print(f"  #{i:3d}  MAE={mae:.3f}{std_str}  RMSE={rmse:.3f}  {desc}")

## 验证集 MAE 演化轨迹 (Val MAE Over Time)

追踪 KEPT 实验的 MAE 如何随演化推进下降。running minimum 是"前沿"——目前达到的最佳精度。
图中叠加 `baseline_registry.py` 的 SOTA 基线水平线，直观看到"逼近并超越 SOTA"的进度。

In [ ]:
# 拉取目标数据集的 SOTA 基线 (用于在图上画参照线)
baselines = {}
try:
    import sys, os
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".")
    from baseline_registry import BASELINE_METRICS
    baselines = {k: v.get("mae") for k, v in BASELINE_METRICS.get(DATASET, {}).items()}
except Exception as e:
    print(f"(未能加载 baseline_registry: {e} — 跳过基线参照线)")

fig, ax = plt.subplots(figsize=(16, 8))

# 剔除 CRASH/PRUNED 只看有效 MAE
valid = df[df["status"].isin(["KEEP", "DISCARD"])].copy().reset_index(drop=True)
baseline_mae = valid.loc[0, "val_mae"]  # 第 0 轮通常是初始构建(冷启动基线)

# 舍弃的实验画成淡灰背景点
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_mae"], c="#cccccc", s=14, alpha=0.5, zorder=2, label="Discarded")

# 保留的实验画成绿色大点
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_mae"], c="#2ecc71", s=55, zorder=4,
           label="Kept", edgecolors="black", linewidths=0.5)

# running best 阶梯线
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_mae = valid.loc[kept_mask, "val_mae"]
running_min = kept_mae.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60", linewidth=2,
        alpha=0.7, zorder=3, label="Running best")

# SOTA / baseline 参照线
palette = ["#e74c3c", "#e67e22", "#9b59b6", "#3498db", "#16a085"]
for (name, mae), color in zip(sorted(baselines.items(), key=lambda kv: kv[1] or 1e9), palette):
    if mae is None:
        continue
    ax.axhline(mae, color=color, ls="--", lw=1.3, alpha=0.8, zorder=1)
    ax.text(0.995, mae, f" {name}={mae}", color=color, fontsize=9, va="bottom", ha="right",
            transform=ax.get_yaxis_transform())

# 标注每个 KEPT 实验的描述
for idx, mae in zip(kept_idx, kept_mae):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, mae), textcoords="offset points", xytext=(6, 6),
                fontsize=8.0, color="#1a7a3a", alpha=0.9, rotation=30, ha="left", va="bottom")

n_total, n_kept = len(df), len(kept_v)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation MAE (masked, lower is better)", fontsize=12)
ax.set_title(f"Darwin-ST Progress on {DATASET}: {n_total} Experiments, {n_kept} Kept", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("已保存到 progress.png")

## Summary Statistics

In [ ]:
# 汇总统计 (Summary stats) — 含 SOTA 差距
kept = df[df["status"] == "KEEP"].copy()
baseline_mae = df.iloc[0]["val_mae"]
best_mae = kept["val_mae"].min()
best_row = kept.loc[kept["val_mae"].idxmin()]

print(f"冷启动基线 MAE (Baseline):   {baseline_mae:.3f}")
print(f"当前最佳 MAE (Best):         {best_mae:.3f}")
print(f"累计改进 (Improvement):      {baseline_mae - best_mae:.3f} "
      f"({(baseline_mae - best_mae) / baseline_mae * 100:.2f}%)")
print(f"最佳实验 (Best experiment):  {best_row['description']}")

# 与 SOTA 的差距 (终极目标判定)
if baselines:
    sota_name = min(baselines, key=lambda k: baselines[k])
    sota_mae = baselines[sota_name]
    gap = best_mae - sota_mae
    status = "✅ 已超越 SOTA!" if gap < 0 else f"还差 {gap:.3f} 追上 {sota_name}"
    print(f"\nSOTA ({sota_name}) MAE = {sota_mae}  →  {status}")
print()

# 每个改进的累计努力
print("每个改进的累计努力 (Cumulative effort per improvement):")
for _, row in kept.reset_index().iterrows():
    desc = str(row["description"]).strip()
    print(f"  #{row['index']:3d}: MAE={row['val_mae']:.3f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# 每个 KEPT 实验的增量 = 相对上一个 KEPT 的 MAE 下降量
# (实验是累积的 — 每个都建立在上一个保留状态之上)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_mae"] = kept["val_mae"].shift(1)
kept["delta"] = kept["prev_mae"] - kept["val_mae"]   # 正值=精度提升

hits = kept.iloc[1:].copy()                          # 丢掉基线(无 delta)
hits = hits.sort_values("delta", ascending=False)    # 改进最大的排前

print(f"{'Rank':>4}  {'Delta':>8}  {'MAE':>8}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.3f}  {row['val_mae']:8.3f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.3f}  {'':>8}  相对基线的累计改进 (TOTAL)")